In [ ]:
# Imports
import os
from dotenv import load_dotenv
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from huggingface_hub import login
from openai.types.audio import transcription
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch





In [ ]:
# Constants

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

In [ ]:
name = "/Users/tarkshya/Work/LLM Engineering/week3HuggingFace/llms/denver_extract.mp3"

In [ ]:
# Sign in to HuggingFace Hub
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

# Opening the File
# No need to make a separate variable to open the file as it opens automatically So don't write Audio_file = open(name,"rb")
# instead just pass the file path directly into the desired model and it will open the file automatically

# Step 1 transcribing The Audio

In [ ]:
from transformers import pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="distil-whisper/distil-small.en",
    device="mps",
    dtype=torch.float16,
    return_timestamps=True,
)
result = pipe(name)
transcription = result["text"]
print(transcription)

In [ ]:
open_src_trans = transcription

# Using OpenAI for transcription

In [ ]:
AUDIO_MODEL = "gpt-4o-mini-transcribe"

# for sending the request to the API , ensure to create the Audio variable we talked out above.
AUDIO_FILE = open(name,"rb")
openai_api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=openai_api_key)

transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL,file = AUDIO_FILE,response_format="text")
print(transcription)


In [ ]:
display(Markdown(open_src_trans))
print("\n\n")
display(Markdown(transcription))

Analyse and Report


In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]

# Making our Model

# Finalising the output


In [ ]:
#tokenizer = AutoTokenizer.from_pretrained(LLAMA)
#tokenizer.pad_token = tokenizer.eos_token
#inputs  = tokenizer.apply_chat_template(messages,return_tensors="pt").to("mps")
#streamer = TextStreamer(tokenizer)
#model = AutoModelForCausalLM.from_pretrained(LLAMA,dtype=torch.bfloat16).to("mps")
#outputs = model.generate(inputs,max_new_tokens=2000,streamer=streamer)

# Since For Llama-3.2-1B-Instruct, apply_chat_template() returns a BatchEncoding, not a raw tensor. Then you're passing the whole BatchEncoding as the first positional argument to generate(), which in our Transformers version isn't being interpreted the way you will expect it to.

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
    add_generation_prompt=True
).to("mps")

streamer = TextStreamer(tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    dtype=torch.bfloat16
).to("mps")

outputs = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=2000,
    streamer=streamer
)


In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))